# ocp-datalake demo

CPU workbench (`HardwareProfile` **default-profile**). The NVIDIA L4 stays on `llama-32-3b-instruct`.

1. `POST /predict` on the in-cluster churn-score gateway.
2. Mint a MaaS `sk-oai-` key and call `/v1/chat/completions` on the Facebook OPT 125M **CPU simulator**.

`scripts/enable-workbench.sh` writes ConfigMap `workbench-demo-env` (live `MAAS_URL`). Do not commit API keys.

In [ ]:
import json, os, ssl, urllib.request

NS = os.environ.get("NOTEBOOK_NAMESPACE", "ocp-datalake")
CHURN_URL = os.environ.get(
    "CHURN_PREDICT_URL", f"http://inference.{NS}.svc:4180/predict"
)
MAAS_URL = os.environ.get("MAAS_URL", "").rstrip("/")
MAAS_MODEL = os.environ.get(
    "MAAS_MODEL", "publishers/llm/models/facebook/opt-125m"
)
CTX = ssl._create_unverified_context()

def sa_token():
    with open("/var/run/secrets/kubernetes.io/serviceaccount/token") as fh:
        return fh.read().strip()

def http_json(method, url, body=None, headers=None, timeout=60):
    data = None if body is None else json.dumps(body).encode()
    req = urllib.request.Request(url, data=data, method=method)
    req.add_header("Content-Type", "application/json")
    for key, value in (headers or {}).items():
        req.add_header(key, value)
    with urllib.request.urlopen(req, context=CTX, timeout=timeout) as resp:
        raw = resp.read().decode()
        return json.loads(raw) if raw else {}

print("CHURN_PREDICT_URL =", CHURN_URL)
print("MAAS_URL          =", MAAS_URL or "(set by scripts/enable-workbench.sh)")
print("MAAS_MODEL        =", MAAS_MODEL)

## 1. Predictive path — `POST /predict`

In-cluster Service `inference` (oauth-proxy skips auth on `/predict`). Same contract as the Web Terminal curl.

In [ ]:
payload = {"tenure": 12, "charges": 70, "support_tickets": 3}
pred = http_json("POST", CHURN_URL, body=payload)
print(json.dumps(pred, indent=2))
assert "churn" in pred, pred

## 2. Models-as-a-Service — chat completions

Uses the workbench ServiceAccount token (`system:authenticated`). The simulator returns random text; that is enough to prove gateway, keys, and quota.

If `MAAS_URL` is empty, run `bash scripts/enable-workbench.sh` on the laptop (writes the live apps domain into ConfigMap `workbench-demo-env`) and restart this kernel.

In [ ]:
assert MAAS_URL, "MAAS_URL missing — run scripts/enable-workbench.sh"
token = sa_token()
key_body = http_json(
    "POST",
    f"{MAAS_URL}/maas-api/v1/api-keys",
    body={"name": "workbench-demo", "subscription": "simulator-free", "expiresIn": "1h"},
    headers={"Authorization": f"Bearer {token}"},
)
api_key = key_body.get("key") or ""
print("key prefix:", (api_key[:10] + "…") if api_key else key_body)
assert api_key.startswith("sk-oai-"), key_body

models = http_json(
    "GET",
    f"{MAAS_URL}/v1/models",
    headers={"Authorization": f"Bearer {api_key}"},
)
print("models:", [m.get("id") for m in models.get("data", [])])

chat = http_json(
    "POST",
    f"{MAAS_URL}/v1/chat/completions",
    body={
        "model": MAAS_MODEL,
        "messages": [{"role": "user", "content": "Say hello in one sentence."}],
        "max_tokens": 16,
    },
    headers={"Authorization": f"Bearer {api_key}"},
)
print(json.dumps(chat, indent=2)[:800])

## 3. Observe

OpenShift AI → **Observe & monitor**:

- **Cluster** (Project All) — stacked CPU by project is the live capacity view.
- **Usage** — MaaS token counters (Limitador). Run `bash scripts/maas-usage-load.sh` if the series are still empty.

Do not print full `sk-oai-` keys in tickets or git.